Por fim, construir conjuntos consolidados de dados e exibições  para responder a perguntas-chave de negócio.

1° Projeto - Visão Comercial e Volume de Produtos

In [0]:
# 1 PROJETO - Entrega 1
from pyspark.sql.functions import col, year, month, countDistinct, count, sum, round

spark.sql("USE CATALOG medalhao")
spark.sql("CREATE DATABASE IF NOT EXISTS gold")

# Carregar tabelas da silver
df_pedidos = spark.table("silver.fat_pedido_total")
df_itens = spark.table("silver.fat_itens_pedidos")
df_produtos = spark.table("silver.dim_produtos")
df_dolar = spark.table("silver.dim_cotacao_dolar")

# Junção em uma só tabela para facilitar o processamento e/ou análise
df_base_comercial = (
    df_itens
    .join(df_pedidos, "id_pedido", "inner") # inner para retornar apenas valores presentes em ambas
    .join(df_produtos, "id_produto", "left") # left para retornar todos os valores na tabela esquerda
    .join(df_dolar, df_pedidos.data_pedido == df_dolar.data_cotacao, "left")
    #.join(df_dolar, col("data_pedido") == col("data_cotacao"),"left")
)

# Entrega 1 - Tabela Principal gold.fat_vendas_comercial
fat_vendas_comercial = (
    df_base_comercial
    .withColumn("ano_venda", year(col("data_pedido")))
    .withColumn("mes_venda", month(col("data_pedido")))

    # Agrupamento por ano, mês e categoria
    .groupBy("ano_venda", "mes_venda", "categoria_produto")
    .agg(
        countDistinct("id_pedido").alias("total_pedidos"), # Contagem de pedidos unitários
        count("id_item").alias("qtd_itens_vendidos"), # Contagem absoluta de itens

        sum("preco_BRL").alias("receita_total_brl_bruta"),
        sum(col("preco_BRL") / col("cotacao_compra")).alias("receita_total_usd_bruta")
    )
    # Calcula o ticket médio em reais (Receita / Total de Pedidos)
    .withColumn("ticket_medio_brl", round(col("receita_total_brl_bruta") / col("total_pedidos"), 2))
    
    # Arredondando os valores financeiros finais
    .withColumn("receita_total_brl", round(col("receita_total_brl_bruta"), 2))
    .withColumn("receita_total_usd", round(col("receita_total_usd_bruta"), 2))

    # Selecionando e ordenando as colunas
    .select(
        "ano_venda", "mes_venda", "categoria_produto", "total_pedidos", 
        "qtd_itens_vendidos", "receita_total_brl", "receita_total_usd", "ticket_medio_brl"
    )
)

# Salvar a tabela Gold
fat_vendas_comercial.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.fat_vendas_comercial")
print("Entrega 1: Tabela gold.fat_vendas_comercial gerada com sucesso!")

Entrega 1: Tabela gold.fat_vendas_comercial gerada com sucesso!


In [0]:
# 1 PROJETO - Entrega 2

df_rankings = (
    df_base_comercial
    .groupBy("nome_produto", "categoria_produto") 
    .agg(count("id_item").alias("quantidade_vendida"))
)

# Top 5 produtos MAIS vendidos
df_top_5_mais = df_rankings.orderBy(col("quantidade_vendida").desc()).limit(5)

# Top 5 produtos MENOS vendidos
df_top_5_menos = df_rankings.orderBy(col("quantidade_vendida").asc()).limit(5)

print("-----TOP 5 PRODUTOS MAIS VENDIDOS-----")
display(df_top_5_mais)

print("\n-----TOP 5 PRODUTOS MENOS VENDIDOS-----")
display(df_top_5_menos)

-----TOP 5 PRODUTOS MAIS VENDIDOS-----


nome_produto,categoria_produto,quantidade_vendida
Secador de Cabelo,beleza_saude,1076
Toalha de Banho,cama_mesa_banho,919
Protetor Solar,beleza_saude,917
Travesseiro,cama_mesa_banho,839
Colar,relogios_presentes,804



-----TOP 5 PRODUTOS MENOS VENDIDOS-----


nome_produto,categoria_produto,quantidade_vendida
Produto Genérico Preto,moveis_quarto,1
Item Básico Premium,fashion_calcados,1
Peça de Reposição Preto,telefonia_fixa,1
Item Básico Verde,industria_comercio_e_negocios,1
Peça de Reposição Plus,fashion_calcados,1


2° Projeto - Satisfação de Clientes e Qualidade de Parceiros

In [0]:
# # 2 PROJETO - Entrega 1
from pyspark.sql.functions import avg, when

df_avaliacoes = spark.table("silver.fat_avaliacoes_pedidos")
df_itens = spark.table("silver.fat_itens_pedidos")
df_produtos = spark.table("silver.dim_produtos")
df_vendedores = spark.table("silver.dim_vendedores")

df_base_qualidade = (
    df_avaliacoes
    .join(df_itens, "id_pedido", "inner")
    .join(df_produtos, "id_produto", "left")
    .join(df_vendedores, "id_vendedor", "left")
)

fat_avaliacoes_clientes = (
    df_base_qualidade
    .groupBy("categoria_produto", "nome_vendedor", "estado")
    .agg(
        count("id_avaliacao").alias("total_avaliacoes"),
        round(avg("nota_avaliacao"), 2).alias("avaliacao_media"),

        # Somatório condicional para avaliações positivas e negativas
        sum(when(col("nota_avaliacao") >= 4, 1).otherwise(0)).alias("total_avaliacoes_positivas"),
        sum(when(col("nota_avaliacao") <= 2, 1).otherwise(0)).alias("total_avaliacoes_negativas")
    )
    # Cálculo do percentual
    .withColumn("percentual_satisfacao", round((col("total_avaliacoes_positivas") / col("total_avaliacoes")) * 100, 2))
)

# Salvando a tabela
fat_avaliacoes_clientes.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.fat_avaliacoes_clientes")
print("Entrega 1: Tabela gold.fat_avaliacoes_clientes gerada com sucesso!")

Entrega 1: Tabela gold.fat_avaliacoes_clientes gerada com sucesso!


In [0]:
# 2 PROJETO - Entrega 2

df_ranking_produtos = (
    df_base_qualidade
    .groupBy("id_produto", "nome_produto", "categoria_produto")
    .agg(
        round(avg("nota_avaliacao"), 2).alias("nota_media"),
        count("id_avaliacao").alias("volume_avaliacoes")
    )
)

# Vendedores
df_ranking_vendedores = (
    df_base_qualidade
    .groupBy("id_vendedor", "nome_vendedor", "estado")
    .agg(
        round(avg("nota_avaliacao"), 2).alias("nota_media"),
        count("id_avaliacao").alias("volume_avaliacoes")
    )
)

# O Produto MAIS bem avaliado
top_produto = df_ranking_produtos.orderBy(col("nota_media").desc(), col("volume_avaliacoes").desc()).limit(1)

# O Produto MENOS bem avaliado
worst_produto = df_ranking_produtos.orderBy(col("nota_media").asc(), col("volume_avaliacoes").desc()).limit(1)

# O Vendedor MAIS bem avaliado
top_vendedor = df_ranking_vendedores.orderBy(col("nota_media").desc(), col("volume_avaliacoes").desc()).limit(1)

# O Vendedor MENOS bem avaliado
worst_vendedor = df_ranking_vendedores.orderBy(col("nota_media").asc(), col("volume_avaliacoes").desc()).limit(1)

# Exibindo os resultados
print("\n-----O PRODUTO MAIS BEM AVALIADO-----")
display(top_produto)

print("\n-----O PRODUTO MENOS BEM AVALIADO-----")
display(worst_produto)

print("\n-----O VENDEDOR MAIS BEM AVALIADO ---")
display(top_vendedor)

print("\n-----O VENDEDOR MENOS BEM AVALIADO-----")
display(worst_vendedor)


-----O PRODUTO MAIS BEM AVALIADO-----


id_produto,nome_produto,categoria_produto,nota_media,volume_avaliacoes
37eb69aca8718e843d897aa7b82f462d,Kit de Ferramentas Dourado,ferramentas_jardim,5.0,15



-----O PRODUTO MENOS BEM AVALIADO-----


id_produto,nome_produto,categoria_produto,nota_media,volume_avaliacoes
0e1fa2aadc04afbf8fb30200aeba06a2,Conjunto de Panelas,utilidades_domesticas,1.0,10



-----O VENDEDOR MAIS BEM AVALIADO ---


id_vendedor,nome_vendedor,estado,nota_media,volume_avaliacoes
48efc9d94a9834137efd9ea76b065a38,Luiz Otávio Abreu,PR,5.0,32



-----O VENDEDOR MENOS BEM AVALIADO-----


id_vendedor,nome_vendedor,estado,nota_media,volume_avaliacoes
8d92f3ea807b89465643c219455e7369,Sra. Fernanda Santos,SP,1.0,8
